# Octopus Architecture Demo

This notebook loads the local Octopus source tree and exercises the AI-native architecture layer described by `AGENTS.md`, source docstrings, and generated registry metadata.

It demonstrates:

- reflection-generated registries
- element specs as structured, introspectable objects
- compilation from specs to runtime tracking objects
- CPU tracking through the existing runtime layer

In [ ]:
const OCTOPUS_ROOT = dirname(@__DIR__)
const OCTOPUS_SRC = joinpath(OCTOPUS_ROOT, "src", "Octopus.jl")

include(OCTOPUS_SRC)
using .Octopus

println("Loaded Octopus from: ", OCTOPUS_SRC)

## Build the Reflection Registry

The registry is generated from Julia types rather than maintained as a separate metadata file.

In [ ]:
registry = build_registry()
summary = summarize_registry(registry)

summary

In [ ]:
?build_registry

## Inspect an ElementSpec

`CrabDispersionSpec` is the structured spec layer. It records physics meaning and supported architecture relationships in metadata that humans, scripts, and agents can inspect. It is not the runtime tracking object.

In [ ]:
spec = CrabDispersionSpec{Float64}(zeta1 = 0.1)

(
    name = name(spec),
    description = description(spec),
    keywords = physics_keywords(spec),
    tracking_methods = supported_tracking_methods(spec),
    analyses = supported_analyses(spec),
    contracts = required_contracts(spec),
)

## Compile the Spec to a Runtime Object

The execution policy owns numerical execution choices. Here the spec is compiled to the existing `CrabDispersion` runtime map using `Symplectic6DMap` and a CPU policy.

In [ ]:
policy = CPUThreadsExecutionPolicy()
runtime_elem = compile_runtime(spec)

typeof(runtime_elem)

## Track One Particle

The runtime layer keeps the tracking representation compact and efficient.

In [ ]:
rep = Phase6DRep(
    [1.0],  # x
    [2.0],  # px
    [3.0],  # y
    [4.0],  # py
    [5.0],  # z
    [6.0],  # pz
)

rep[1]

In [ ]:
track!(rep, (runtime_elem,), 1, CPUThreadsBackend)

rep[1]

## Docstrings as Agent-Readable Metadata

Octopus uses Julia docstrings as part of the source-of-truth knowledge layer. Public architectural APIs should explain their role clearly enough that a human or AI agent can inspect them without reading every implementation detail.

In [ ]:
doc_targets = [
    :build_registry,
    :compile_runtime,
    :CrabDispersionSpec,
    :ThinCrabCavitySpec,
    :ThinCrabCavity,
    :CrabDispersion,
    :CPUThreadsExecutionPolicy,
    :CUDAExecutionPolicy,
    :TrackingTask,
    :execute!,
]

[(target, Base.Docs.doc(getfield(Octopus, target)) !== nothing) for target in doc_targets]

In a live notebook, Julia's help mode can also show the same source docstrings directly. For example, run `?CrabDispersionSpec`, `?compile_runtime`, or `?build_registry` in a notebook cell.

In [ ]:
Base.Docs.doc(CrabDispersionSpec)

## Define a Task

A task bundles an element sequence, tracking method, policy, validation contracts, and analyses into one workflow object. The sequence is stored at the specification layer first; it is compiled to runtime tracking objects only when executed.

In [ ]:
line = (
    CrabDispersionSpec{Float64}(zeta1 = 0.1),
    MomentumDispersionSpec{Float64}(eta1 = 0.2),
    XYCouplingSpec{Float64}(r1 = 0.01),
)

task = TrackingTask(line; policy = policy)

(
    elements = name.(task.elements),
    policy = typeof(task.policy),
    contracts = task.contracts,
    analyses = task.analyses,
)

## Execute the Task

`execute!` compiles the task's element specs into a runtime element sequence, selects the backend from the execution policy, and tracks an existing phase-space representation in place. The same task can be saved, inspected, and executed later on one or many particles.

In [ ]:
task_rep = Phase6DRep(
    [1.0, 0.2],
    [2.0, 0.1],
    [3.0, 0.3],
    [4.0, 0.4],
    [5.0, 0.5],
    [6.0, 0.6],
)

execute!(task, task_rep; turns = 2)

[task_rep[i] for i in keys(task_rep)]

In [ ]:
task_rep

In [ ]:
keys(task_rep)

In [ ]:
task_rep[1]

In [ ]:
task_rep[2]

## Thin Crab Cavity

`ThinCrabCavity{N}` uses a Julia type parameter for the harmonic count `N`.

In [ ]:
cavity_spec = ThinCrabCavitySpec{2}(
    1.0e8;
    strengthX = (1.0, 0.5),
    strengthY = (0.25, 0.125),
    phase = (0.0, 0.1),
)

cavity = compile_runtime(cavity_spec)

(
    type = typeof(cavity),
    first_harmonic_strength_x = cavity[1, 1],
    second_harmonic_strength_y = cavity[2, 2],
    tracked_particle = track_particle(Symplectic6DMap, cavity, 1.0, 2.0, 3.0, 4.0, 0.01, 6.0),
)

In [ ]:
construction_help(LorentzBoostSpec)

In [ ]:
?construction_help

In [ ]:
?LorentzBoostSpec

In [ ]:
construction_help(ElementSpec{:lorentz_boost})

In [ ]:
parameter_schema(ElementSpec{:lorentz_boost})

In [ ]:
parameter_schema(ElementSpec{:thin_crab_cavity})

In [ ]:
typeof(ans)

In [ ]:
using DataFrames

function metadata_table(spec::NamedTuple)
    DataFrame([
        (
            parameter = String(name),
            required = info.required,
            unit = isempty(info.unit) ? "—" : info.unit,
            default = repr(info.default),
            meaning = info.meaning,
        )
        for (name, info) in pairs(spec)
    ])
end


In [ ]:
tcc=parameter_schema(ElementSpec{:thin_crab_cavity})
metadata_table(tcc)